In [81]:
import argparse
import json
import re
import sys
from biom import load_table

control_pattern_default = r"(blank|control|neg|ntc|water|reagent)"
table_path = "../data/V4_stool/V4_stool.biom"
ambiguities_path = "../data/V4_stool/V4_stool.biom.ambiguities"

# Compile control regex
is_control = re.compile(control_pattern_default, re.I).search

# Load table
try:
    table = load_table(table_path)
except Exception as e:
    sys.exit(f"[error] failed to load BIOM table '{table_path}': {e}")

# Load ambiguities mapping
try:
    with open(ambiguities_path) as f:
        ambig = json.load(f)
except Exception as e:
    sys.exit(f"[error] failed to read ambiguities JSON '{ambiguities_path}': {e}")

# Current sample IDs present in table
present = set(table.ids(axis="sample"))

keep = set()
sample_to_idx = {sample_id: i for i, sample_id in enumerate(table.ids(axis="sample"))}
idx_to_sample = {i: sample_id for sample_id, i in sample_to_idx.items()}
sample_depths = table.sum(axis='sample')

# Helper: total observation count (sequencing depth) for a sample
def depth(sample_id: str) -> float:
    # table.sum(axis='sample', ids=[id]) returns a 1-element ndarray
    return float(sample_depths[sample_to_idx[sample_id]])



In [76]:

print(len(present), "samples before filtering")

# for sample_id in present:
#     print(f"sample_id: {sample_id}")
#     print(depth(sample_id))
#     break

42500 samples before filtering


In [ ]:
# import random
# def test_depth(sample_id: str) -> float:
#     return random.randint(1000, 10000)

In [ ]:
# kept items from replicate sets
clip_count = 5000
kept_from_replicates = 0
replicate_total = len([run for runs in ambig.values() for run in runs])

for canon, runs in ambig.items():
    # print(f"canon: {canon}, runs: {runs}")
    
    if is_control(canon):
        print(f"  skipping control run: {canon}")
        continue
    avail = [run for run in runs if run in present]
    if not avail:
        print(f"  no available samples for run: {canon}")
        continue
    best = max(avail, key=depth)
    if depth(best) < clip_count:
        print(f"  skipping low-depth run: {best} (depth: {depth(best)})")
        continue
    # print(best)
    keep.add(best)
    kept_from_replicates += 1
print(f"kept {kept_from_replicates} samples out of {replicate_total} ambiguous samples.")
    




  skipping low-depth run: 10564.DC6MEYNB.134305 (depth: 3513.0)
  skipping low-depth run: 10564.O9G6DQH2.134305 (depth: 2745.0)
  skipping low-depth run: 10317.X00215889.147023 (depth: 28.0)
  skipping low-depth run: 10317.000001525.131452 (depth: 131.0)
  skipping low-depth run: 10317.X00214414.147023 (depth: 51.0)
  skipping low-depth run: 10564.QCMUUEDW.134305 (depth: 3417.0)
  skipping low-depth run: 2086.1325500812A.131614 (depth: 9.0)
  skipping low-depth run: 2086.1325904517.129439 (depth: 15.0)
  skipping low-depth run: 10317.000020791.131424 (depth: 85.0)
  skipping low-depth run: 10317.000038111.131418 (depth: 4315.0)
  skipping low-depth run: 10317.X00215244.156882 (depth: 80.0)
  skipping low-depth run: 2086.1325401895.129439 (depth: 4.0)
  skipping low-depth run: 10564.74ZDKQVK.134305 (depth: 4977.0)
  skipping low-depth run: 10564.YZL1C941.134305 (depth: 4883.0)
  skipping low-depth run: 10564.8HDXIBAS.134305 (depth: 3127.0)
  skipping low-depth run: 10564.CE3XUVX2.134305

In [83]:
# subtract all ambiguous sample from present, then add back kept ones
ambig_runs = {run for runs in ambig.values() for run in runs}
unambiguous = present - ambig_runs

print(f"{len(present)} total samples")
print(f"{len(ambig_runs)} ambiguous samples")
print(f"{len(unambiguous)} unambiguous samples")

for sample_id in unambiguous:
    if is_control(sample_id):
        print(f"skipping control run: {sample_id}")
        continue
    if depth(sample_id) < clip_count:
        print(f"  skipping low-depth run: {sample_id} (depth: {depth(sample_id)})")
        continue
    keep.add(sample_id)

print(f"{len(keep)} total samples after filtering.")

42500 total samples
6744 ambiguous samples
35756 unambiguous samples
  skipping low-depth run: 894.YY1830.lane2.NoIndex.L002.128416 (depth: 13.0)
  skipping low-depth run: 10317.000012201.128816 (depth: 2703.0)
  skipping low-depth run: 12906.Milk.T6.171.132803 (depth: 4816.0)
  skipping low-depth run: 1958.Antibioticday5.10.86.128677 (depth: 1003.0)
  skipping low-depth run: 10317.000053363.132132 (depth: 3069.0)
  skipping low-depth run: 10093.s3d42p6.127811 (depth: 1134.0)
  skipping low-depth run: 1958.Preantibiotic.4.62.128677 (depth: 1414.0)
  skipping low-depth run: 10317.000042933.132132 (depth: 3035.0)
  skipping low-depth run: 11113.1122.127587 (depth: 25.0)
  skipping low-depth run: 10317.000011077.128816 (depth: 3734.0)
  skipping low-depth run: 10317.000016182.128434 (depth: 2560.0)
  skipping low-depth run: 10317.000040346.128247 (depth: 1498.0)
  skipping low-depth run: 10317.000015195.128434 (depth: 4.0)
skipping control run: 11914.St.neg.GLD065.155591
  skipping low-de

In [84]:
from biom.util import biom_open
filter_fn = lambda val, id_, md: id_ in keep
new_table = table.filter(filter_fn, inplace=False)
out_path = "cleaned_table.biom"
with biom_open(out_path, "w") as f:
    new_table.to_hdf5(f, "cleaned table")
print(f"Wrote cleaned BIOM table to: {out_path}")

Wrote cleaned BIOM table to: cleaned_table.biom


In [ ]:
# import numpy as np
# from biom.table import Table
# data = np.asarray([[0, 0, 1], [1, 3, 42]])
# table = Table(data, ['O1', 'O2'], ['S1', 'S2', 'S3'],
#               [{'full_genome_available': True},
#                {'full_genome_available': False}],
#               [{'sample_type': 'a'}, {'sample_type': 'a'},
#                {'sample_type': 'b'}])
# print(table)
# kept = set()
# kept.add('S1')
# kept.add('S2')
# filter_fn = lambda val, id_, md: id_ in kept
# new_table = table.filter(filter_fn, inplace=False)
# print(new_table)

# Constructed from biom file
#OTU ID	S1	S2	S3
O1	0.0	0.0	1.0
O2	1.0	3.0	42.0
# Constructed from biom file
#OTU ID	S1	S2
O1	0.0	0.0
O2	1.0	3.0


In [5]:
samples = ["10317.000030031.129073", "10469.Meta.p05.wk08.130664", "894.UY150.lane3.NoIndex.L003.128416", "10532.03a27c57.cf57.417a.90fe.5b9d5ffb32c0.128763"]

mapping = {c: ".".join(c.split(".")[:-1]) for c in samples}
mapping

{'10317.000030031.129073': '10317.000030031',
 '10469.Meta.p05.wk08.130664': '10469.Meta.p05.wk08',
 '894.UY150.lane3.NoIndex.L003.128416': '894.UY150.lane3.NoIndex.L003',
 '10532.03a27c57.cf57.417a.90fe.5b9d5ffb32c0.128763': '10532.03a27c57.cf57.417a.90fe.5b9d5ffb32c0'}

In [1]:
from biom import load_table
from skbio import TreeNode

table = load_table("../data/v4_stool/v4_stool_cleaned_filtered_biom/feature-table.biom")
tree  = TreeNode.read("../crawford.tre")  # or your own tree

otu_ids = set(table.ids(axis='observation'))          # OTU/ASV IDs from BIOM
tree_ids = set(tip.name for tip in tree.tips())       # tip labels from tree

print("n_otus_in_table:", len(otu_ids))
print("n_tips_in_tree:", len(tree_ids))
print("intersection:", len(otu_ids & tree_ids))

# sanity check: does tree fully cover table?
print("all_otus_covered?", otu_ids.issubset(tree_ids))

n_otus_in_table: 1875457
n_tips_in_tree: 365
intersection: 0
all_otus_covered? False


In [3]:
from skbio import TreeNode

tree = TreeNode.read("tree.nwk")

for i, tip in enumerate(tree.tips()):
    print(tip.name)
    if i == 4:   # just print first 5
        break


GB-GCA-003568775.1-MWMI01000008.1
GB-GCA-001552015.1-CP010514.1
GB-GCA-000008085.1-AE017199.1
TAGCCGCACCCCAAGTGGTAGTCATTATTATTGGGCTTAAAGTGTTCGTAGCCGGGCCTGAAAGTCCGCTGTGAAATCCAAGCGCTCAAC
TACCCGCGCCACGAGTGGTGATCGCGATTATTGGGCCTAAAGGGTTCGTAGCCGGTTTGGCAAGTTCCTGGTGAAATCTTTCAGCTAACTGAAAGGCGTG


In [10]:
count150 = 0
for i, tip in enumerate(tree.tips()):
    if len(tip.name) == 150:
        count150 += 1

print("n_tips_with_150_char_names:", count150)

   

n_tips_with_150_char_names: 8927883


In [11]:
from biom import load_table
table = load_table("../data/v4_stool/v4_stool_cleaned_filtered_biom/feature-table.biom")
otu_ids = set(table.ids(axis='observation'))          # OTU/ASV IDs from BIOM
tree_ids = set(tip.name for tip in tree.tips())       # tip labels from tree

print("n_otus_in_table:", len(otu_ids))
print("n_tips_in_tree:", len(tree_ids))
print("intersection:", len(otu_ids & tree_ids))

# sanity check: does tree fully cover table?
print("all_otus_covered?", otu_ids.issubset(tree_ids))

n_otus_in_table: 1875457
n_tips_in_tree: 23450268
intersection: 1875457
all_otus_covered? True


In [ ]:
from biom import load_table
import unifrac
import numpy as np

# 1. Load table
table = load_table("../data/v4_stool/v4_stool_cleaned_filtered_biom/feature-table.biom")

# Ordered OTU/ASV IDs (150bp sequences) – KEEP AS ARRAY, NOT set()
otu_ids = table.ids(axis='observation')           # length = n_features
sample_ids = table.ids(axis='sample')             # length = n_samples

print("n_features:", len(otu_ids))
print("n_samples :", len(sample_ids))

# 2. Pick two samples, e.g. first two columns
samp0_vec = table[:, 0].toarray().flatten()       # shape (n_features,)
samp1_vec = table[:, 1].toarray().flatten()       # shape (n_features,)

print("samp0_vec shape:", samp0_vec.shape)
print("samp1_vec shape:", samp1_vec.shape)

# 3. Compute UniFrac distance between these two samples
TREE_FP = "tree.nwk"  # path to your (possibly huge) GG2 ASV tree

dist01 = unifrac.unweighted_dense_pair(
    otu_ids,      # ordered list/array of ASV IDs
    samp0_vec,    # counts for sample 0
    samp1_vec,    # counts for sample 1
    TREE_FP
)

print("UniFrac(s0, s1) =", dist01)


n_features: 1875457
n_samples : 36668
samp0_vec shape: (1875457,)
samp1_vec shape: (1875457,)


AttributeError: module 'unifrac' has no attribute 'unweighted_sparse_pair'

In [50]:
# read metadata file txt 
import pandas as pd
import difflib

def load_metadata(metadata_fp: str, id_col: str) -> pd.DataFrame:
    """Load metadata file as Pandas DataFrame."""
    try:
        meta = pd.read_csv(metadata_fp, sep="\t", dtype={id_col: str})
    except Exception as e:
        raise ValueError(f"Failed to read metadata file '{metadata_fp}': {e}")
    return meta

df = load_metadata("../data/test_set/human_test_set_v4_10317_stool_cleaned_filtered/sample-metadata.txt", id_col="#SampleID")


#read from tsv
a = open("../data/test_set/human_test_set_v4_10317_stool_cleaned_filtered/sample-metadata.txt").readlines()[0]
b = a.strip().split('\t')
print(b)

# # get headers
# columns = df.columns.tolist()
# print('number of metadata columns:', len(columns))
# print("Metadata columns:", "\n".join(columns))

# query = "Colonic"

# col = [s for s in columns if query in s]
# print("Direct matches for column name:", col)

# matches = difflib.get_close_matches(query, columns, n=3, cutoff=0.2)

# print("Close matches for column name:", matches)

['#SampleID', 'specialized_diet_unspecified', 'mental_illness_type_anorexia_nervosa', 'artificial_sweeteners', 'last_travel', 'bowel_movement_frequency', 'physical_specimen_remaining', 'vegetable_frequency', 'olive_oil', 'allergic_to_peanuts', 'ready_to_eat_meals_frequency', 'public', 'fruit_frequency', 'allergic_to_i_have_no_food_allergies_that_i_know_of', 'seafood_frequency', 'types_of_plants', 'pku', 'host_weight', 'title', 'mental_illness_type_unspecified', 'whole_eggs', 'multivitamin', 'milk_cheese_frequency', 'dog', 'diabetes', 'other_supplement_frequency', 'longitude', 'autoimmune', 'sugary_sweets_frequency', 'thyroid', 'non_food_allergies_drug_eg_penicillin', 'dna_extracted', 'teethbrushing_frequency', 'migraine', 'cat', 'vivid_dreams', 'host_taxid', 'chickenpox', 'level_of_education', 'lung_disease', 'fungal_overgrowth', 'alcohol_types_beercider', 'kidney_disease', 'non_food_allergies_beestings', 'non_food_allergies_poison_ivyoak', 'host_height', 'sample_type', 'env_package', 

/var/folders/8s/djsz761n16x6mh8gpddd5dh40000gn/T/ipykernel_8384/3114817814.py:8: DtypeWarning: Columns (1,9,13,30,43,44,50,51,53,62,73,90,102,110,121,123,140,145,149,153,168,178) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(metadata_fp, sep="\t", dtype={id_col: str})


In [ ]:
from collections import defaultdict
# read metadata file txt 
import pandas as pd
import difflib

def load_metadata(metadata_fp: str, id_col: str) -> pd.DataFrame:
    """Load metadata file as Pandas DataFrame."""
    try:
        meta = pd.read_csv(metadata_fp, sep="\t", dtype={id_col: str})
    except Exception as e:
        raise ValueError(f"Failed to read metadata file '{metadata_fp}': {e}")
    return meta

#df = load_metadata("../data/test_set/human_test_set_v4_10317_stool_cleaned_filtered/sample-metadata.txt", id_col="#SampleID")
df = load_metadata("../data/v4_stool/v4_stool_cleaned_filtered_biom/sample-metadata.tsv", id_col="#SampleID")

# get headers
columns = df.columns.tolist()
print("Metadata columns:", "\n".join(columns))
freq = defaultdict(int)
for x in df['qiita_study_id']:
    freq[x] += 1
results = sorted([(s, f) for s, f in freq.items()], key=lambda s: s[1], reverse=True)
main = results[:1]
others = results[1:]
print(main)
print(sum(f for _, f in others))

print(results)

# The problem, we need a train set that has collection invariance / batch effects that DO NOT EXIST in the test set.  Does that also mean the train set and the test set must also all be human and stool samples so all else is equal?  i.e the only difference that exists is collection noise?

# the assumption is that different study ids correspond to different collection processes

#


Metadata columns: #SampleID
sample_type
host_subject_id
qiita_study_id
physical_specimen_location
collection_timestamp
[(10317, 24626)]
15389
[(10317, 24626), (10184, 1924), (894, 1859), (2086, 1638), (2014, 1017), (11113, 969), (11349, 909), (12906, 687), (11888, 645), (10323, 605), (2401, 492), (10532, 474), (1998, 279), (11360, 274), (1939, 257), (16036, 230), (11123, 221), (10315, 193), (11402, 188), (2182, 159), (11625, 158), (10093, 157), (11212, 154), (10057, 138), (1924, 133), (10724, 130), (13010, 130), (10916, 118), (11914, 112), (11129, 106), (10564, 102), (10064, 93), (1880, 93), (1056, 92), (11404, 80), (11947, 77), (10342, 58), (11333, 56), (13338, 53), (11076, 50), (10469, 40), (10902, 39), (10904, 32), (10704, 30), (10461, 30), (15602, 29), (10407, 27), (10786, 24), (10522, 13), (10563, 12), (1684, 3)]


Unsupervised training 

study,human?,species_label, train,test_features, samplesN, comments
10317 36000 samples
10184,yes, host_scientific_name, yes,geo_loc_name, 1924, good for test we should add many human studies to claim invariance to
12906, yes, host_common_name, yes, geo_loc_name, 1017, good
2401,host_scientific_name, yes, geo_loc_name host_age, 492, good
10532, yes, host_scientific_name, yes, geo_loc_name, 474, good
1998, yes, host_scientific_name, yes, geo_loc_name only usa age, 279, good
1939, yes, host_scientific_name, yes, geo_loc_name age, gastrointest_disord a lot more, 257, good
16036, yes, host_scientific_name, yes, geo_loc_name, 230, good


#classify geo_loc_name we want better performance with embeddings than raw
#classify study_id we want worse performance with embeddings than raw

Plan
10317 + bottom half of the table above for unsupervised training of the embeddings encoder
Then test on the top half of the table for supervised tasks such as geo_loc_name and study_id classification
Use the rest of 10317 for supervised tasks such as age regression and chrons disease classification

Then punch out and say fuck it


invariant Holdouts 
2086, yes, host_scientific_name, yes, age geo_loc_name, 1638, good
2014, yes, host_scientific_name, yes, age geo_loc_name, 1017, good
11888, yes, host_common_name, yes, geo_loc_name host_age, 645, good


Core components:

📘 Encoding: Transformer-based sample embedding aligned to UniFrac phylogenetic distances

🔬 Datasets: American Gut Project (Qiita 10317, Deblur V4), plus several additional Qiita studies used to capture batch-effect variability.
All samples are human stool samples.

Study IDs included in the training dataset:

10184 (dark blue)

10532 (light blue)

2401 (orange)

16036 (light orange)

12906 (green)

1998 (light green)

1939 (red)

Holdouts for classification evaluation
(Each holdout study provides unseen samples with their own metadata distributions; all are human stool samples.)

2086 — host_scientific_name: yes, age: yes, geo_loc_name: yes, n = 1638, good

2014 — host_scientific_name: yes, age: yes, geo_loc_name: yes, n = 1017, good

11888 — host_common_name: yes, host_age: yes, geo_loc_name: yes, n = 645, good

⚙️ Preprocessing: BIOM → (sample_id, reads, 150bp nucleotides) triplet decomposition to (sample_id, embedding) for fine-grained modeling

📊 Evaluation: Distance correlation with UniFrac, downstream classifier accuracy on real data.

Goal:
Create uniform, biologically grounded representations of 16S microbiome data that enable cross-study compatibility and increased classifer accuracy on downstream health prediction tasks.


In [4]:
5350/30

178.33333333333334

In [2]:
'10317.000009111'.startswith('10517')

False

In [ ]:
#candidates country_of_birth ibd host_age ibd_diagnosis_refined acid_reflux

# max of host age filter for number and convert to int
import string


ages = [float(x) for x in df['host_age'] if x != "not provided" and x != "not collected"]
ages = [x for x in ages if x >= 0 and x <= 120] 

def host_age_label(x: string) -> int:
    
    if x >= 0 and x <= 120:
        return 1
    else:
        return 0

print(ages)
print(len(ages))
print("average host age:", sum(ages)/len(ages))
# print(ages)
print("max host age:", max(ages))
print("min host age:", min(ages))

ibd = [x for x in df['ibd'] if x == "Diagnosed by a medical professional (doctor, physician assistant)" or x == 'Diagnosed by an alternative medicine practitioner' or x == 'Self-diagnosed']
len(ibd)
print('len ibd:', len(ibd))

acid_reflux = [x for x in df['acid_reflux'] if x == "Diagnosed by a medical professional (doctor, physician assistant)" or x == 'Diagnosed by an alternative medicine practitioner' or x == 'Self-diagnosed']

def acid_reflux_label(x: str) -> int:
    if x == "Diagnosed by a medical professional (doctor, physician assistant)" or x == 'Diagnosed by an alternative medicine practitioner' or x == 'Self-diagnosed':
        return 1
    else:
        return 0

print("len acid reflux:", len(acid_reflux))
# country_of_birth = [x for x in df['country_of_birth']]
# print("set country of birth:", set(country_of_birth))
# print("len country of birth:", len(country_of_birth))


df[['country_of_birth', 'ibd', 'host_age', 'ibd_diagnosis_refined', 'acid_reflux']]



[74.9, 45.6, 64.3, 1.7, 30.5, 56.7, 36.5, 46.3, 58.0, 43.6, 66.8, 51.0, 61.4, 8.2, 78.0, 38.0, 59.1, 25.6, 71.3, 50.8, 59.8, 51.9, 34.2, 39.3, 37.0, 46.3, 67.1, 52.4, 70.2, 37.0, 41.9, 44.1, 25.1, 52.5, 3.9, 69.2, 46.6, 0.4, 73.0, 41.4, 60.6, 34.5, 37.7, 35.9, 40.7, 33.2, 41.1, 69.6, 34.9, 69.4, 48.1, 74.5, 40.3, 25.0, 48.0, 47.9, 63.8, 51.3, 72.4, 48.9, 53.6, 70.6, 52.1, 56.2, 75.9, 60.8, 27.0, 4.1, 56.6, 52.5, 26.6, 67.7, 36.1, 51.8, 60.5, 46.4, 72.2, 49.5, 45.1, 6.5, 51.9, 38.1, 48.4, 50.5, 30.6, 36.9, 35.9, 71.4, 10.3, 54.0, 27.2, 68.6, 60.3, 15.9, 33.1, 4.6, 55.5, 70.3, 30.5, 36.1, 55.0, 56.8, 48.5, 55.0, 74.0, 60.1, 27.4, 28.9, 74.2, 49.9, 40.6, 59.0, 33.1, 41.3, 30.6, 52.9, 22.0, 32.5, 47.2, 60.8, 70.8, 93.7, 65.3, 1.5, 43.1, 21.9, 0.9, 52.7, 51.6, 24.5, 53.9, 67.5, 15.2, 42.0, 41.9, 39.6, 24.3, 30.4, 57.0, 48.7, 79.8, 29.9, 61.7, 55.7, 57.5, 29.4, 13.6, 76.5, 79.4, 57.4, 27.1, 54.3, 40.6, 48.3, 69.4, 59.2, 51.9, 53.8, 30.2, 61.3, 2.4, 55.2, 50.9, 30.9, 7.9, 67.3, 64.2, 31.1, 21

,country_of_birth,ibd,host_age,ibd_diagnosis_refined,acid_reflux
0,United Kingdom,I do not have this condition,74.9,not provided,not provided
1,United States,I do not have this condition,45.6,not provided,I do not have this condition
2,Canada,I do not have this condition,64.3,not provided,I do not have this condition
3,Philippines,I do not have this condition,1.7,not provided,I do not have this condition
4,Belgium,I do not have this condition,30.5,not provided,I do not have this condition
...,...,...,...,...,...
23243,United States,I do not have this condition,54.7,not provided,I do not have this condition
23244,United States,I do not have this condition,9.1,not provided,not provided
23245,Philippines,I do not have this condition,46.1,not provided,not provided
23246,Panama,I do not have this condition,37.2,not provided,I do not have this condition


In [62]:
# load pandas on PcOA embeddings
import pandas as pd
df = pd.read_csv("../data/test_set/human_test_set_v4_10317_stool_cleaned_filtered/small_human_test_set/pcoa_unifrac_128d.tsv", sep="\t", index_col=0)

df.head()

# show sample 1 embedding arrray
print(df.iloc[0].values)
df.head()


[-2.20747675e-01  1.01932855e-01  7.41596528e-02  8.75519202e-02
 -1.90833758e-02  2.47534024e-02  9.67651988e-02 -3.13807901e-02
 -1.54947870e-02 -1.81431494e-02  9.68213572e-03 -2.57824883e-02
 -9.15328796e-02  2.04690221e-02 -5.93849175e-03 -1.17177395e-02
  6.26750728e-02 -1.11077071e-02 -4.69859062e-03 -2.16770951e-02
  1.90508616e-02 -1.98003763e-02  1.37795701e-02 -1.05463970e-02
 -1.05396218e-02 -5.47750804e-03  9.24149478e-03  1.17238843e-02
 -9.26230336e-03  1.77815472e-02  4.06860740e-02  3.13511661e-02
 -1.20219263e-02 -1.19398717e-02  6.26406513e-03  1.37613832e-02
 -1.05669150e-02 -7.51584217e-03 -4.77312175e-02  4.60423873e-03
  3.75573442e-02 -4.36971293e-02  2.99854918e-02 -5.72942549e-03
  3.58053949e-02 -1.78136864e-02 -1.00838852e-02 -3.68480492e-02
 -3.48050882e-02  1.60953648e-02  2.97098428e-02  3.48991119e-03
  3.19833119e-03  6.16646673e-03  1.42088756e-02 -3.02274809e-02
 -9.08470875e-03  8.52611223e-03  1.69752685e-03  1.28922013e-03
 -2.95772677e-02  3.39190

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC119,PC120,PC121,PC122,PC123,PC124,PC125,PC126,PC127,PC128
#SampleID,,,,,,,,,,,,,,,,,,,,,
10317.000068952,-0.220748,0.101933,0.074160,0.087552,-0.019083,0.024753,0.096765,-0.031381,-0.015495,-0.018143,...,-0.018134,0.004183,0.000176,0.007817,0.014380,-0.002185,-0.011253,0.020210,0.001296,-0.002229
10317.000010121,-0.073468,0.056386,0.075175,0.035734,-0.048489,-0.016126,-0.075775,0.012261,0.005084,-0.016214,...,0.012519,0.003936,-0.003869,0.003937,-0.003871,0.020602,-0.011327,0.000938,-0.023318,-0.012972
10317.000074841,0.317487,0.052352,0.179231,0.064255,0.048005,0.009229,-0.031858,0.092285,0.041188,-0.048499,...,0.034030,0.021550,-0.014371,0.018587,-0.002063,-0.000050,-0.000950,0.012535,0.034088,-0.034518
10317.000116304,-0.044913,-0.194850,-0.055408,-0.099490,0.059005,-0.028117,0.020583,-0.004798,0.022232,-0.018132,...,0.012976,-0.001030,0.000318,0.009470,0.007801,-0.019035,-0.002901,0.042147,0.023617,0.022861
10317.000066998,0.175066,0.026144,-0.000806,-0.073150,-0.033313,-0.053787,-0.035493,-0.071881,0.030153,0.039513,...,-0.003180,-0.006272,0.014779,0.003074,0.008558,0.002440,-0.003548,0.009268,0.015277,0.031801


In [ ]:
import pandas as pd
from skbio import DistanceMatrix
from skbio.stats.ordination import pcoa
import matplotlib.pyplot as plt

# 1. Load UniFrac
dm_df = pd.read_csv("unifrac.tsv", sep="\t", index_col=0)
dm = DistanceMatrix(dm_df.values, ids=dm_df.index)

# 2. PCoA
pcoa_results = pcoa(dm)
coords = pcoa_results.samples  # rows = samples

# 3. Load metadata and add study_id if needed
meta = pd.read_csv("metadata.tsv", sep="\t").set_index("sample-id")

if "study_id" not in meta.columns:
    meta["study_id"] = meta.index.str.split(".").str[0]

coords = coords.join(meta, how="inner")

# 4. Plot: color by study_id, shape by geo_loc_name
plt.figure(figsize=(7, 6))

markers = {name: marker for name, marker in zip(
    coords["geo_loc_name"].unique(),
    ['o', 's', '^', 'D', 'P', 'X', '*']
)}

for (geo_loc), group in coords.groupby("geo_loc_name"):
    marker = markers.get(geo_loc, 'o')
    sc = plt.scatter(
        group["PC1"],
        group["PC2"],
        marker=marker,
        alpha=0.7,
        c=pd.Categorical(group["study_id"]).codes,
    )

plt.xlabel(f"PC1 ({pcoa_results.proportion_explained['PC1']*100:.1f}% var)")
plt.ylabel(f"PC2 ({pcoa_results.proportion_explained['PC2']*100:.1f}% var)")
plt.title("PCoA on UniFrac – color=study_id, shape=geo_loc_name")
plt.tight_layout()
plt.show()
